# Davinci Notebook 12 — SFT Training, Chat Formatting & Tool Traces

Canonical chat serialization, assistant-only target masking, host-compatible tool-call message structure, training/validation splitting, and a fine-tuning loop. The exact token spellings are imported from the repository SFT module and must match the frozen SentencePiece tokenizer. Tool observations are context, not assistant targets.


In [ ]:
import json, torch, torch.nn.functional as F, hashlib
from pathlib import Path
from torch.utils.data import Dataset,DataLoader


## 1. Canonical message structure and serialization
Use a consistent serialization strategy for training and inference. The assistant tool-call payload follows the strict JSON protocol in `src/davinci/tool_call.py`: exactly `name` and `arguments`. The host executes only registered tools; tool observations use the `tool` role and are not assistant-authored targets.


In [ ]:
import sys, subprocess
REPO_DIR=Path("/content/Davinci")
if not (REPO_DIR/".git").exists():
    subprocess.run(["git","clone","--depth","1","https://github.com/keyhmoham/Davinci.git",str(REPO_DIR)],check=True)
else:
    subprocess.run(["git","-C",str(REPO_DIR),"pull","--ff-only"],check=True)
sys.path.insert(0,str(REPO_DIR))
from src.davinci import DavinciForCausalLM
from src.davinci.data_pipeline import validate_dataset_manifest, verify_dataset_artifact, sha256_file
from src.davinci.sft import (
    SPECIAL_TOKENS, build_masked_example, causal_sft_loss, collate_masked,
    serialize_messages, split_examples, tool_call_content, tool_result_content,
)
REPO_COMMIT=subprocess.check_output(["git","-C",str(REPO_DIR),"rev-parse","HEAD"],text=True).strip()
print("Imported SFT helpers from",REPO_COMMIT)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import sentencepiece as spm

# Explicit inputs from the preceding data/pretraining stages.
SFT_JSONL_PATH=Path("/content/drive/MyDrive/Davinci/sft_data/processed/farsinstruct_chat_sft.jsonl")
TOKENIZER_MODEL=Path("/content/drive/MyDrive/Davinci/tokenizer/davinci_sentencepiece.model")
SFT_MANIFEST_PATH=Path("/content/drive/MyDrive/Davinci/sft_data/processed/farsinstruct_manifest.json")
BASE_MODEL_DIR=Path("/content/drive/MyDrive/Davinci/pretraining_runs/run_001/pretrained_model")
PRETRAINING_RUN_MANIFEST=BASE_MODEL_DIR.parent/"run_manifest.json"
OUTPUT_DIR=Path("/content/drive/MyDrive/Davinci/sft_runs/run_001")
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
for required in (SFT_JSONL_PATH,TOKENIZER_MODEL):
    if not required.is_file(): raise FileNotFoundError(f"Required SFT input is missing: {required}")
if not (BASE_MODEL_DIR/"config.json").is_file() or not (BASE_MODEL_DIR/"model.pt").is_file():
    raise FileNotFoundError(f"Exported pretrained model directory is incomplete: {BASE_MODEL_DIR}")
if not PRETRAINING_RUN_MANIFEST.is_file():
    raise FileNotFoundError(f"Pretraining run manifest is missing: {PRETRAINING_RUN_MANIFEST}")
PRETRAINING_MANIFEST=json.loads(PRETRAINING_RUN_MANIFEST.read_text(encoding="utf-8"))
if PRETRAINING_MANIFEST.get("tokenizer_sha256")!=sha256_file(TOKENIZER_MODEL):
    raise ValueError("SFT tokenizer does not match the tokenizer used by the base pretraining run.")
PRETRAINING_RUN_MANIFEST_SHA256=sha256_file(PRETRAINING_RUN_MANIFEST)
if not SFT_MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"SFT dataset manifest is missing: {SFT_MANIFEST_PATH}")
SFT_MANIFEST=json.loads(SFT_MANIFEST_PATH.read_text(encoding="utf-8"))
validate_dataset_manifest(SFT_MANIFEST,intended_purpose="instruction")
verify_dataset_artifact(SFT_JSONL_PATH,SFT_MANIFEST,hash_field="chat_jsonl_sha256")

tokenizer=spm.SentencePieceProcessor(model_file=str(TOKENIZER_MODEL))
if tokenizer.pad_id()<0: raise ValueError("Tokenizer must define a PAD token for padded SFT batches.")
for token in SPECIAL_TOKENS.values():
    if tokenizer.piece_to_id(token)==tokenizer.unk_id():
        raise ValueError(f"Tokenizer is missing required special token: {token}")

def canonicalize_messages(raw_messages):
    if not isinstance(raw_messages,list): raise ValueError("messages must be a list")
    output=[]
    for item in raw_messages:
        if not isinstance(item,dict): raise ValueError("each message must be an object")
        role=item.get("role")
        if role=="assistant" and (item.get("type")=="tool_call" or "tool_call" in item):
            call=item.get("tool_call",item)
            content=tool_call_content(call.get("name"),call.get("arguments"))
        elif role=="tool":
            if not isinstance(item.get("trusted"),bool):
                raise ValueError("tool observations must carry an explicit trusted boolean")
            if item.get("name") and not str(item.get("content","")).startswith(SPECIAL_TOKENS["tool_result"]):
                content=tool_result_content(item["name"],item.get("content"))
            else:
                content=item.get("content","")
        else:
            content=item.get("content","")
        if not isinstance(content,str): raise ValueError("message content must be text after tool serialization")
        output.append({"role":role,"content":content})
    return output

examples=[]; rejected=[]
with SFT_JSONL_PATH.open(encoding="utf-8") as handle:
    for row_number,line in enumerate(handle):
        if not line.strip(): continue
        try:
            record=json.loads(line)
            messages=canonicalize_messages(record.get("messages"))
            examples.append({
                "id":record.get("id",row_number),
                "source_document_id":record.get("source_document_id"),
                "split":record.get("split"),
                "messages":messages,
            })
        except Exception as exc:
            rejected.append({"row":row_number,"reason":str(exc)})
if not examples: raise ValueError("No valid message examples were loaded from the configured JSONL.")
if any(item["split"] not in {"train","validation"} for item in examples):
    raise ValueError("Every SFT example must carry the reviewed train/validation split from the dataset builder.")
print({"valid_examples":len(examples),"rejected_examples":len(rejected),"tokenizer_vocab":tokenizer.get_piece_size()})
(OUTPUT_DIR/"rejected_rows.json").write_text(json.dumps(rejected,ensure_ascii=False,indent=2),encoding="utf-8")


In [ ]:
SPECIAL=SPECIAL_TOKENS
example_trace=[
 {"role":"user","content":"What is 19 + 23?"},
 {"role":"assistant","content":tool_call_content("add",{"left":19,"right":23})},
 {"role":"tool","content":tool_result_content("add",{"sum":42})},
 {"role":"assistant","content":"42"},
]
serialized_example=serialize_messages(example_trace)
assert SPECIAL["tool_call"] in serialized_example and SPECIAL["tool_result"] in serialized_example
assert SPECIAL["end"] in serialized_example
print(serialized_example)

## 2. Assistant-only masking dataset
The caller must supply a tokenizer that can tokenize the serialized messages and identify the token boundaries. This template masks non-assistant tokens with `-100`; adapt it to the actual tokenizer's special token IDs and turn boundaries.

In [ ]:
messages=[
 {"role":"user","content":"سلام"},
 {"role":"assistant","content":"درود"},
]
# Token spellings are imported from the same repository module used by the model runtime.
example=build_masked_example(messages,tokenizer)
print("tokens:",len(example["input_ids"]),"assistant targets:",sum(x!=-100 for x in example["labels"]))

## 3. Dataset split and collator
Split by example ID or source document before training; avoid overlapping related examples across splits. This basic collator pads and uses labels=-100 for padding.

In [ ]:
train_examples=[item for item in examples if item["split"]=="train"]
val_examples=[item for item in examples if item["split"]=="validation"]
if not train_examples or not val_examples:
    raise ValueError("SFT train/validation split is empty; rebuild the dataset with more reviewed examples.")
print("train:",len(train_examples),"validation:",len(val_examples))
batch=collate_masked([build_masked_example(x["messages"],tokenizer) for x in train_examples[:4]],pad_id=tokenizer.pad_id())

## 4. SFT loop (generic model interface)
Assumes model(input_ids) returns logits `[B,T,V]`. This is a baseline loop; it omits packed-example masking, multi-GPU, and production scheduler/checkpoint policies.

In [ ]:
def sft_train_step(model,batch,optimizer,device,clip=1.0):
 ids,labels=[x.to(device) for x in batch]
 model.train()
 optimizer.zero_grad(set_to_none=True)
 output=model(ids,labels=labels)
 loss=output["loss"] if isinstance(output,dict) else causal_sft_loss(output,labels)
 if loss is None: raise ValueError("Batch contains no assistant next-token targets")
 loss.backward()
 torch.nn.utils.clip_grad_norm_(model.parameters(),clip)
 optimizer.step()
 return float(loss.detach())

print("SFT step is compatible with DavinciForCausalLM(input_ids, labels=labels).")

## 5. Dataset audit before fine-tuning
- Check source licenses and redistribution conditions.
- Deduplicate against pretraining and held-out data.
- Review generated reasoning for correctness and leakage.
- Remove personally identifying or confidential content.
- Keep tool traces structured and separate observations from model-authored messages.
- Evaluate separately on Persian, English, bilingual and tool-use tasks.


## 6. Tool-trace protocol alignment
Before training, parse every assistant tool-call JSON body with the same strict protocol as the inference runtime, validate tool names/arguments against the tool manifest, and check that each tool result corresponds to a preceding call. If `<|tool_call|>` is a dedicated token, detect it as a structural boundary and pass only the following JSON body to the parser. Do not train the model to emit host-only secrets, stack traces, or unredacted tool output. Tool-call assistant messages are supervised targets; tool observations remain context only in the assistant-only masking template above.


In [ ]:
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
model=DavinciForCausalLM.from_pretrained(BASE_MODEL_DIR,map_location="cpu")
if model.config.vocab_size!=tokenizer.get_piece_size():
    raise ValueError("Base model vocabulary does not match the selected tokenizer.")
model.to(DEVICE)

encoded_train=[build_masked_example(item["messages"],tokenizer) for item in train_examples]
encoded_val=[build_masked_example(item["messages"],tokenizer) for item in val_examples]
PAD_ID=tokenizer.pad_id()
BATCH_SIZE=2
NUM_EPOCHS=1
LEARNING_RATE=2e-5
WEIGHT_DECAY=0.01
train_loader=DataLoader(encoded_train,batch_size=BATCH_SIZE,shuffle=True,
    collate_fn=lambda items:collate_masked(items,PAD_ID))
val_loader=DataLoader(encoded_val,batch_size=BATCH_SIZE,shuffle=False,
    collate_fn=lambda items:collate_masked(items,PAD_ID))
optimizer=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)

def evaluate_sft(model,loader):
    was_training=model.training; model.eval(); values=[]
    try:
        with torch.no_grad():
            for ids,labels in loader:
                output=model(ids.to(DEVICE),labels=labels.to(DEVICE))
                if output["loss"] is not None: values.append(float(output["loss"].cpu()))
    finally:
        model.train(was_training)
    if not values: raise ValueError("Validation produced no assistant target tokens.")
    return sum(values)/len(values)

history=[]
for epoch in range(NUM_EPOCHS):
    losses=[]
    for batch in train_loader:
        losses.append(sft_train_step(model,batch,optimizer,DEVICE))
    val_loss=evaluate_sft(model,val_loader)
    history.append({"epoch":epoch+1,"train_loss":sum(losses)/len(losses),"validation_loss":val_loss})
    print(history[-1])

model_dir=OUTPUT_DIR/"model"
model.save_pretrained(model_dir)
def file_sha256(path):
    digest=hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda:handle.read(1024*1024),b""):
            digest.update(chunk)
    return digest.hexdigest()
base_config_hash=file_sha256(BASE_MODEL_DIR/"config.json")
base_weights_hash=file_sha256(BASE_MODEL_DIR/"model.pt")
tokenizer_hash=file_sha256(TOKENIZER_MODEL)
data_hash=file_sha256(SFT_JSONL_PATH)
sft_manifest_hash=file_sha256(SFT_MANIFEST_PATH)
torch.save({
    "model":model.state_dict(),"optimizer":optimizer.state_dict(),"history":history,
    "repo_commit":REPO_COMMIT,"base_config_sha256":base_config_hash,"base_weights_sha256":base_weights_hash,
    "pretraining_run_manifest_sha256":PRETRAINING_RUN_MANIFEST_SHA256,
    "tokenizer_sha256":tokenizer_hash,"dataset_sha256":data_hash,"dataset_manifest_sha256":sft_manifest_hash,
    "model_config":model.config.__dict__,
},OUTPUT_DIR/"sft_checkpoint.pt")
manifest={
    "repo_commit":REPO_COMMIT,"base_config_sha256":base_config_hash,"base_weights_sha256":base_weights_hash,
    "tokenizer_sha256":tokenizer_hash,"dataset_sha256":data_hash,"dataset_manifest_sha256":sft_manifest_hash,
    "pretraining_run_manifest_sha256":PRETRAINING_RUN_MANIFEST_SHA256,
    "train_examples":len(encoded_train),"validation_examples":len(encoded_val),
    "batch_size":BATCH_SIZE,"epochs":NUM_EPOCHS,"learning_rate":LEARNING_RATE,
    "weight_decay":WEIGHT_DECAY,"device":DEVICE,"history":history,
    "limitations":["single-process baseline","no gradient accumulation","no DPO/preference optimization"],
}
(OUTPUT_DIR/"run_manifest.json").write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding="utf-8")
print("Saved SFT model and manifest to",OUTPUT_DIR)
